In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, month, avg, count, row_number, to_date
from pyspark.sql.window import Window

In [3]:
spark = (SparkSession.builder
    .appName("trendtrackers-pa5")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate())

print("Spark version:", spark.version)

Spark version: 4.0.2


In [4]:
!wget -O sample10k.tsv https://raw.githubusercontent.com/sjsu-cs131-spring26/trendtrackers-engagement-socialmedia/main/data/sample10k.tsv

--2026-04-18 04:41:58--  https://raw.githubusercontent.com/sjsu-cs131-spring26/trendtrackers-engagement-socialmedia/main/data/sample10k.tsv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3875902 (3.7M) [text/plain]
Saving to: ‘sample10k.tsv’

sample10k.tsv       100%[===================>]   3.70M  --.-KB/s    in 0.07s   

2026-04-18 04:41:59 (52.6 MB/s) - ‘sample10k.tsv’ saved [3875902/3875902]



In [5]:
df = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", "\t")
    .csv("sample10k.tsv"))

In [6]:
df.printSchema()
df.show(5, truncate=False)
print("Rows:", df.count())
print("Columns:", df.columns)

root
 |-- date: string (nullable = true)
 |-- original_text: string (nullable = true)
 |-- url: string (nullable = true)
 |-- author_hash: string (nullable = true)
 |-- language: string (nullable = true)
 |-- primary_theme: string (nullable = true)
 |-- english_keywords: string (nullable = true)
 |-- sentiment: string (nullable = true)
 |-- main_emotion: string (nullable = true)
 |-- secondary_themes: string (nullable = true)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------+--------------------------------------------------+----------------------------------------+-------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [7]:
from pyspark.sql.functions import col, when

clean_df = df.filter(
    col("primary_theme").isNotNull() &
    col("date").isNotNull() &
    col("sentiment").isNotNull()
)

clean_df = clean_df.filter(
    col("sentiment").rlike(r"^-?\d+(\.\d+)?$")
)

clean_df = clean_df.withColumn("sentiment", col("sentiment").cast("double"))

clean_df.select("date", "primary_theme", "sentiment").show(5, truncate=False)

+------------------------+-------------+---------+
|date                    |primary_theme|sentiment|
+------------------------+-------------+---------+
|2024-12-01T00:38:21.000Z|Entertainment|-0.47    |
|2024-12-01T00:38:16.000Z|Entertainment|-0.75    |
|2024-12-01T01:03:04.000Z|Social       |-0.56    |
|2024-12-01T00:30:06.000Z|Politics     |-0.58    |
|2024-12-01T00:10:56.000Z|People       |-0.68    |
+------------------------+-------------+---------+
only showing top 5 rows


In [8]:
window_spec = Window.partitionBy("primary_theme").orderBy(col("sentiment").desc())

ranked_df = clean_df.withColumn(
    "rank_in_theme",
    row_number().over(window_spec)
)

ranked_df.select("primary_theme", "sentiment", "rank_in_theme").show(20, truncate=False)

+-------------+---------+-------------+
|primary_theme|sentiment|rank_in_theme|
+-------------+---------+-------------+
|Business     |0.86     |1            |
|Business     |0.82     |2            |
|Business     |0.8      |3            |
|Business     |0.76     |4            |
|Business     |0.74     |5            |
|Business     |0.73     |6            |
|Business     |0.72     |7            |
|Business     |0.71     |8            |
|Business     |0.7      |9            |
|Business     |0.7      |10           |
|Business     |0.69     |11           |
|Business     |0.68     |12           |
|Business     |0.66     |13           |
|Business     |0.64     |14           |
|Business     |0.63     |15           |
|Business     |0.59     |16           |
|Business     |0.59     |17           |
|Business     |0.59     |18           |
|Business     |0.58     |19           |
|Business     |0.55     |20           |
+-------------+---------+-------------+
only showing top 20 rows


In [9]:
top3_per_theme = ranked_df.filter(col("rank_in_theme") <= 3)

top3_per_theme.orderBy("primary_theme", "rank_in_theme") \
    .show(50, truncate=False)

+------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------+----------------------------------------+--------+--------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+------------+----------------+-------------+
|date                    |original_text                                                                                                                                                                                            

In [10]:
from pyspark.sql.functions import col, when, month, avg, count, to_timestamp

trend_df = (df
    .filter(col("primary_theme").isNotNull())
    .filter(col("sentiment").rlike(r"^-?\d+(\.\d+)?$"))
    .filter(col("date").rlike(r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d{3}Z$"))
    .withColumn("sentiment_num", col("sentiment").cast("double"))
    .withColumn("date_ts", to_timestamp(col("date"), "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'"))
)

monthly_trend = (trend_df
    .withColumn("month", month("date_ts"))
    .groupBy("month")
    .agg(
        avg("sentiment_num").alias("avg_sentiment"),
        count("*").alias("row_count")
    )
    .orderBy("month")
)

monthly_trend.show(truncate=False)

+-----+--------------------+---------+
|month|avg_sentiment       |row_count|
+-----+--------------------+---------+
|12   |0.030066777963272017|7787     |
+-----+--------------------+---------+

